# Execution notebook — Segmentation (UNet / MONAI)

**Type:** execution notebook.

## Purpose

Train UNet (MONAI), save best checkpoint, export masks to `02_dataset/results_*`. Adapted from `01_academic/.../FetalVeinSegmentationUS.ipynb`. Metrics: `04_evaluation/evaluation.ipynb`.

## Imports

MONAI/PyTorch training stack; `%run` postprocessing library for pairing.


## Configuration

Change **only** the parameters below to switch experiments.

| `DATASET_FOLDER` | `RESULTS_FOLDER` | `MODEL_NAME` (example) |
|------------------|-------------------|----------------------|
| `images` | `results_original` | `best_metric_model_original.pth` |
| `images_pp_1` | `results_pp_1` | `best_metric_model_pp_1.pth` |
| `images_pp_2` | `results_pp_2` | `best_metric_model_pp_2.pth` |
| `images_pp_3` | `results_pp_3` | `best_metric_model_pp_3.pth` |
| `images_pp_4` | `results_pp_4` | `best_metric_model_pp_4.pth` |
| `images_pp_5` | `results_pp_5` | `best_metric_model_pp_5.pth` |

In [ ]:
# ==========================================================
# SEGMENTATION EXPERIMENT CONFIGURATION
# ==========================================================

EXPERIMENT_ID = 1
DATASET_FOLDER = "images_pp_1"
RESULTS_FOLDER = "results_pp_1"
MODEL_NAME = "best_metric_model_pp_1.pth"

# Train / validation / test split (same as lecturer notebook)
TRAIN_COUNT = 110
VAL_COUNT = 20

## Project paths

Files in `02_dataset/images` and `02_dataset/labels` are **not modified**. Predictions are written only to `RESULTS_FOLDER`.

In [ ]:
from pathlib import Path

# Project root from postprocessing_common (%run in the next cell) or resolved locally:
try:
    PROJECT_ROOT
except NameError:
    def find_project_root(start_path: Path) -> Path:
        for candidate_path in [start_path, *start_path.parents]:
            if (candidate_path / "02_dataset").is_dir():
                return candidate_path
        raise FileNotFoundError("02_dataset/ not found.")

    PROJECT_ROOT = find_project_root(Path.cwd().resolve())

DATA_DIR = PROJECT_ROOT / "02_dataset"
IMAGES_DIR = DATA_DIR / DATASET_FOLDER
LABELS_DIR = DATA_DIR / "labels"
SAVE_MODELS_DIR = DATA_DIR / "Save_Models"
RESULTS_DIR = DATA_DIR / RESULTS_FOLDER

for output_dir in (SAVE_MODELS_DIR, RESULTS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

print(f"Experiment: {EXPERIMENT_ID}")
print(f"Root: {PROJECT_ROOT}")
print(f"Images: {IMAGES_DIR}")
print(f"Labels: {LABELS_DIR}")
print(f"Model: {SAVE_MODELS_DIR / MODEL_NAME}")
print(f"Predictions: {RESULTS_DIR}")

## Shared library (pairing and validation)

Loads `03_postprocessing/postprocessing_common.ipynb` via `%run` so image–label pairing matches evaluation.

In [ ]:
%run ../03_postprocessing/postprocessing_common.ipynb

import glob


def build_data_dicts(images_folder: Path) -> list:
    """Pairs each image with its label using the original identifier (postprocessing_common)."""
    images_list = sorted(glob.glob(str(images_folder / "*.png")))
    if len(images_list) == 0:
        raise FileNotFoundError(f"No .png images in {images_folder}")

    pares = []
    for image_path in images_list:
        label_path = resolve_label_path(Path(image_path).name, LABELS_DIR)
        pares.append({"image": image_path, "label": str(label_path)})

    return pares


# Pre-training pairing validation
pairing_errors = validate_pairing_folder(IMAGES_DIR, LABELS_DIR)
if pairing_errors:
    raise RuntimeError(
        "Image–label pairing failed. First errors:\n"
        + "\n".join(pairing_errors[:10])
    )

data_dicts = build_data_dicts(IMAGES_DIR)
train_files = data_dicts[:TRAIN_COUNT]
val_files = data_dicts[TRAIN_COUNT : TRAIN_COUNT + VAL_COUNT]
test_files = data_dicts[TRAIN_COUNT + VAL_COUNT :]

print(f"Total pairs: {len(data_dicts)}")
print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")
print(f"Device is set in the model section.")

## Setup imports

(Block adapted from lecturer `FetalVeinSegmentationUS.ipynb`.)

In [ ]:
import os
import shutil
import tempfile
import time
import matplotlib.pyplot as plt
from monai.config import print_config
from monai.data import DataLoader, decollate_batch
from monai.handlers.utils import from_engine
from monai.losses import DiceLoss
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.data import CacheDataset, DataLoader, Dataset, decollate_batch, list_data_collate
from monai.transforms import (
    Activations,
    Activationsd,
    AsDiscrete,
    AsDiscreted,
    Compose,
    Invertd,
    LoadImaged,
    MapTransform,
    NormalizeIntensityd,
    Orientationd,
    RandFlipd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandSpatialCropd,
    Spacingd,
    EnsureTyped,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    SpatialPadd,
)
from monai.utils import set_determinism
import glob
import torch
import pdb
from PIL import Image

print_config()

## Set deterministic training for reproducibility

In [ ]:
set_determinism(seed=0)

## Setup transforms for training, validation, and testing

In [ ]:
train_transform = Compose(
    [
        LoadImaged(keys=["image", "label"], image_only=False),
        EnsureChannelFirstd(keys=["image", "label"]),
        ScaleIntensityRanged(
            keys=["image", "label"],
            a_min=0,
            a_max=255,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
    ]
)

val_transform = Compose(
    [
        LoadImaged(keys=["image", "label"], image_only=False),
        EnsureChannelFirstd(keys=["image", "label"]),
        ScaleIntensityRanged(
            keys=["image", "label"],
            a_min=0,
            a_max=255,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
    ]
)

test_transform = Compose(
    [
        LoadImaged(keys=["image", "label"], image_only=False),
        EnsureChannelFirstd(keys=["image", "label"]),
        ScaleIntensityRanged(
            keys=["image", "label"],
            a_min=0,
            a_max=255,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
    ]
)

## Create Datasets

In [ ]:
train_ds = Dataset(data=train_files, transform=train_transform)
train_loader = DataLoader(
    train_ds, batch_size=10, shuffle=True, num_workers=0, collate_fn=list_data_collate
)

val_ds = Dataset(data=val_files, transform=val_transform)
val_loader = DataLoader(
    val_ds, batch_size=5, shuffle=False, num_workers=0, collate_fn=list_data_collate
)

test_ds = Dataset(data=test_files, transform=test_transform)
test_loader = DataLoader(
    test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=list_data_collate
)

## Layers, blocks, networks and loss functions

In [ ]:
max_epochs = 150
val_interval = 1
VAL_AMP = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

loss_function = DiceLoss(
    smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True
)
optimizer = torch.optim.Adam(model.parameters(), 1e-2)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

dice_metric = DiceMetric(include_background=True, reduction="mean")
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")

post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

## Execute a typical PyTorch training process

Validation `DiceMetric` selects the best checkpoint only (lecturer logic), not the final evaluation report.

In [ ]:
best_metric = -1
best_metric_epoch = -1
best_metrics_epochs_and_time = [[], [], []]
epoch_loss_values = []
val_values = []

best_model_path = SAVE_MODELS_DIR / MODEL_NAME
file_path_ultimo_modelo = SAVE_MODELS_DIR / f"last_{MODEL_NAME}"

total_start = time.time()

for epoch in range(max_epochs):
    epoch_start = time.time()
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step_start = time.time()
        step += 1
        inputs, labels = (
            batch_data["image"].to(device),
            batch_data["label"].to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        print(
            f"{step}/{len(train_ds) // train_loader.batch_size}"
            f", train_loss: {loss.item():.4f}"
            f", step time: {(time.time() - step_start):.4f}"
        )

    lr_scheduler.step()

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)

    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data["image"].to(device),
                    val_data["label"].to(device),
                )
                val_outputs = model(val_inputs)

                loss_val = loss_function(val_outputs, val_labels)

                val_outputs = torch.sigmoid(val_outputs)
                val_outputs = val_outputs > 0.5

                dice_metric(y_pred=val_outputs, y=val_labels)
                dice_metric_batch(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            val_values.append(loss_val)
            metric_batch = dice_metric_batch.aggregate()

            dice_metric.reset()
            dice_metric_batch.reset()

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                best_metrics_epochs_and_time[0].append(best_metric)
                best_metrics_epochs_and_time[1].append(best_metric_epoch)
                best_metrics_epochs_and_time[2].append(time.time() - total_start)
                torch.save(model.state_dict(), best_model_path)
                print("saved new best metric model")
            print(
                f"current epoch: {epoch + 1} current mean dice: {metric:.4f}"
                f"\nbest mean dice: {best_metric:.4f}"
                f" at epoch: {best_metric_epoch}"
            )
    print(f"time consuming of epoch {epoch + 1} is: {(time.time() - epoch_start):.4f}")
    torch.save(model.state_dict(), file_path_ultimo_modelo)

total_time = time.time() - total_start
print(f"Training finished in {total_time:.1f}s. Best model: {best_model_path}")

## Plot the learning curves

In [ ]:
plt.figure("train", (12, 6))
plt.subplot(1, 2, 1)
plt.title("Train Average Loss")
x = [i + 1 for i in range(len(epoch_loss_values))]
y = epoch_loss_values
plt.xlabel("epoch")
plt.plot(x, y, color="red")
plt.subplot(1, 2, 2)
plt.title("Val Mean Loss")
val_values_cpu = [t.to("cpu") for t in val_values]
x = [val_interval * (i + 1) for i in range(len(val_values_cpu))]
y = val_values_cpu
plt.xlabel("epoch")
plt.plot(x, y, color="green")
plt.show()

## Test-set inference — save predicted masks

Predictions are saved under `02_dataset/{RESULTS_FOLDER}/` using the **same filename** as the input image. Nunca dentro de `images` nem `images_pp_*`.

In [ ]:
threshold = 0.5

list_imgs = []
list_labels = []
saved_prediction_paths = []

model.load_state_dict(torch.load(best_model_path))
model.eval()
with torch.no_grad():
    for test_data in test_loader:
        test_inputs, test_labels, meta_imagem = (
            test_data["image"].to(device),
            test_data["label"].to(device),
            test_data["image_meta_dict"],
        )
        test_outputs = model(test_inputs)
        test_outputs = torch.sigmoid(test_outputs)
        test_outputs = test_outputs > threshold

        seg = test_outputs[0, 0, :, :].cpu().numpy()
        gt = test_labels[0, 0, :, :].cpu().numpy()

        saved_prediction_paths.append(seg)
        list_labels.append(gt)
        list_imgs.append(test_inputs[0, 0, :, :].cpu().numpy())

        input_path = Path(meta_imagem["filename_or_obj"][0])
        output_path = RESULTS_DIR / input_path.name
        seg_uint8 = (seg * 255).astype("uint8")
        Image.fromarray(seg_uint8).save(output_path)

print(f"Saved {len(saved_prediction_paths)} mask(s) to {RESULTS_DIR}")

## Visualize the results for one image

In [ ]:
if len(list_imgs) > 0:
    indice = min(9, len(list_imgs) - 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Image")
    plt.imshow(list_imgs[indice][:, :], cmap="gray")
    plt.subplot(1, 3, 2)
    plt.title("Label")
    plt.imshow(list_labels[indice][:, :], cmap="gray")
    plt.subplot(1, 3, 3)
    plt.title("Predicted")
    plt.imshow(saved_prediction_paths[indice][:, :], cmap="gray")
    plt.tight_layout()
    plt.show()

## Conclusions

Training and mask export complete for the configured experiment. Run `04_evaluation/evaluation.ipynb` for metrics with and without `pos_process()`.
